# Атака на подгруппы в протоколе Диффи-Хеллмана на эллиптических кривых

## Группа точек на эллиптической кривой

Пора бы уже привыкнуть к понятию группы, но давайте повторим, что это такое:

Группа - это множество $G$, на котором определена операция $\cdot$, которая преобразует два элемента $a$ и $b$ в другой элемент, который обозначается $a\cdot b$ или $ab$, у него (множества) есть следующие свойства:

+ Замыкание\
    $\forall a,b \in G, a\cdot b \in G$

+ Ассоциативность\
    $\forall a,b,c \in G, (a\cdot b)\cdot c=a\cdot (b\cdot c)$
    
+ Нейтральный элемент\
    $\exists e \in G: \forall a \in G,\ e\cdot a=a\cdot e=a$
    
+ Обратный элемент\
    $\forall\ a\ \exists\ b: a\cdot b=b\cdot a= e$, $b$ обозначается как $a^{-1}$
    
давайте посмотрим, что такое эллиптические кривые и как они встраиваются в определение группы.

Эллиптическая кривая с действительными коэффициентами может быть задана  уравнением Вейерштрассе в короткой форме
$$y^2=x^3 + Ax + B$$
для некоторых $A$, $B$.
Вот как она выглядит:

![Elliptic Curve in R](rcurve.png)

Общее уравнение Вейерштрассе:
$$y^2+a_1xy+a_3y=x^3+a_2x^2+a_4x+a_6$$

Арифметика эллиптических кривых достаточно проста. Если провести прямую через две точки на кривой $P$ и $Q$, то прямая пересечёт кривую в третьей точке $R$. В группе эллиптической кривой задано следующее уравнение:
$$P+Q+R=0$$

![Elliptic Curve with Points](simpleaddition.png)

Но что такое ноль?  Мы представляем точку на бесконечности (не настоящая точка) в виде нуля.  Принимая это во внимание становится очевидным, что чтобы получить обратный элемент какой-либо точки $P$, необходимо провести прямую вертикальную линию через эту точку:

![Inverse P](negpointoncurve.png)

Итак $P+(-P)=0$

Как складывать точки? Нужно провести прямую через точки $P$ и $Q$, найти третье пересечение прямой с кривой (точка $R$, которая всегда будет определена на гладкой кривой, если $P\ne-Q$)  и найти обратную этому пересечению точку:
$$P+Q=-R$$

![Addition](additionexample.png)

Что если  $P=Q$, т.е. нужно удвоить точку? Тогда надо провести касательную к кривой в $P$ и использовать обратную точку к другому пересечению прямой с кривой (за исключением случая, когда $Py=0$, тогда $P=-P$ и $P+P=0$). Всегда можно найти пересечение, если дискриминант эллиптической кривой $\Delta=-16(4A^3+27B^2)$ не равен нулю.

![Point Doubling](pointdoubling.png)

Итак для точки $P=(Px,Py)$ существует нейтральный элемент - точка на бесконечности, обратный элемент - точка $-P=(Px,-Py)$. Очевидно, что множество замкнуто по операции сложения (либо сложение даст точку на кривой, либо точку на бесконечности). Единственное, что остается доказать, это ассоциативность. Доказательство легкое, но достаточно длинное. Если хотите, можете посмотреть геометрическое доказательство [здесь](https://ocw.mit.edu/courses/18-783-elliptic-curves-spring-2021/resources/mit18_783s21_notes2/), а алгебраическое [здесь](https://cocalc.com/share/public_paths/a6a1c2b188bd61d94c3dd3bfd5aa73722e8bd38b). Также группа очевидно коммутативна: $P+Q=Q+P$.

Поскольку есть сложение, можно ввести операцию умножения точки на кривой на скаляр: $$Q=kP=\overbrace{P+...+P}^{k}$$

Эллиптические кривые в поле действительных чисел $\mathbb{R}$ или поле рациональных чисел $\mathbb{Q}$ не очень полезны для нас в смысле шифрования. Поэтому кривые задаются в полях по модулю простого числа $F_p$. Вот как выглядит эллиптическая кривая с теми же параметрами, но заданная в $F_{197}$.

![Elliptic Curve in Field Modulo Prime Number](modularec.png)

Группы эллиптических кривых могут быть использованы в асимметричной криптографии. Пусть есть точка $G$, которая назывется генератор и она имеет порядок $n$ - большое простое число ($nG=0$). Тогда закрытый ключ - это какое-то случайное целое число $k \in \{1,n-1\}$, а открытый ключ - $Q=kG$. Вычисление $k$ при известных $Q$ и $G$ - тяжелая задача и она называется проблемой дискретного логарифмирования в $E(F_p)$.

## ECDH
Диффи-Хеллман в группе точек эллиптической кривой (ECDH) достаточно прост:

1. Алиса генерирует закрытый ключ $a$, открытый ключ $A=aG$, отправляет открытый ключ Бобу

2. Боб генерирует закрытый ключ $b$, открытый ключ $B=bG$, отправляет Алисе открытый ключ

3. Алиса и Боб вычисляют $abG=K=baG$  и используют его или какую-то функцию от него в качестве ключа для симметричного шифрования

Какой вообще смысл использовать ECDH, когда есть RSA и DH? У эллиптических кривых много преимуществ:

+ Для каждого $p$  существует много групп $E(F_p)$ с разными мощностями (порядками)

+ Операция на группу (сложение точек) может быть крайне эффективной (меньше вычислений $\implies$ меньше энергопотребление)

+ Существуют способы создания групп с любой желаемой мощностью

+ Считается, что представление элементов группы непрозрачное, что делает $E(F_p)$ хорошим кандидатом в "black-box группы", к которым применимы только **общие групповые алгоритмы**.

Последнее свойство крайне важно. Если верить NIST, для достижения уровня безопасности в 128 бит нужно использовать модули RSA или DH длиной 3072 бита. Чтобы получить такой же уровень безопасности в группе Эллиптической Кривой нужен модуль $p$ длиной всего лишь 256 бит, что **сильно**  уменьшает необходимое количество вычислений для каждой операции.

## Задание

Атака на подгруппы DH - это обобщенная атака, которая зависит только от наличия удобного разложения порядка группы. Поэтому, давайте попробуем решить это задание как задание на подгруппы в обобщенном DH, прежде чем мы перейдем к особенностям эллиптических кривых. Дана точка на эллиптической кривой с порядком, который можно разложить на малые (<18 бит) множители и один большой, но меньший чем 30 бит. Используя те же принципы, что и в заданиях на подгруппы DH и алгоритм Полларда, нужно найти решение. Важно не забывать, что нужно заключить точку, которую вы пытаетесь логарифмировать, в подгруппу. Также я крайне рекомендую сначала попробовать решить задачу не для неизвестной $k$, а для какой-либо сгенерированной локально для проверки решения.

In [10]:
class IllegalScalar(Exception):
    pass
class EllipticCurve:
    def __init__(self, p, a, b):
        self.p = p
        self.a = a % p
        self.b = b % p

    def __eq__(self, oc):
        return self.p == oc.p and self.a == oc.a and self.b == oc.b

    def __str__(self):
        a,b=self.a,self.b
        a_str= '' if a==0 else str(a)+'*x' if a<0 else '+'+str(a)+'*x'
        b_str= '' if b==0 else str(b) if a<0 else '+'+str(b)

        return f"E(GF({self.p})) для y**2=x**3"+a_str+b_str
    
    def __repr__(self):
        return self.__str__()
class Point:
    def __init__(self, curve, x, y,is_identity=False):
        self.x = x % curve.p
        self.y = y % curve.p
        self.curve = curve
        self._is_identity=is_identity

    def is_identity(self):
        return self._is_identity

    def __eq__(self, Q):
        return (self._is_identity and Q._is_identity) or ( self.x == Q.x and self.y == Q.y and self.curve == Q.curve)

    def __str__(self):
        if self._is_identity:
            return "Точка на Бесконечности на "+str(self.curve)
        return "({0},{1}) на ".format(self.x, self.y)+str(self.curve)
    
    def __repr__(self):
        return self.__str__()

    def __neg__(self):
        return Point(self.curve, self.x, self.curve.p-self.y,self._is_identity)

    def __add__(self, Q):
        if self.is_identity():

            return Q
        if Q.is_identity():

            return self
        if self.x == Q.x and self.y == self.curve.p-Q.y:
            return Point(self.curve, 0, 1,True)
        if self == Q:
            m = (((3*pow(self.x, 2, self.curve.p)+self.curve.a) %
                  self.curve.p)*pow(2*self.y, self.curve.p-2, self.curve.p)) % self.curve.p
        else:
            m = ((Q.y-self.y)*pow(Q.x -
                                  self.x, self.curve.p-2, self.curve.p)) % self.curve.p
        x3 = (pow(m, 2, self.curve.p)-self.x-Q.x) % self.curve.p
        y3 = (m*(self.x-x3)-self.y) % self.curve.p
        return Point(self.curve, x3, y3)

    def __mul__(self, k):
        if type(k) != int:
            raise IllegalScalar
        negate=False
        if k < 0:
            k=-k
            negate=True
        if k == 0:
            return Point(self.curve, 0, 1,True)
        bp = Point(self.curve, 0, 1,True)
        if negate:
            P = -self
        else:
            P=self
        i = 0
        while k != 0:
            if k & 1 != 0:
                bp = bp+P
            k >>= 1
            P = P+P
        return bp

    def __rmul__(self, k):
        return self.__mul__(k)


In [11]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключение к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1347))
       
    def recv_until(self,symb=b'\n>'):
        """Получение сообщений от сервера, по умолчанию до первого приглашения"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def getChallenge(self,show=True):
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка при декодировании Юникода. Попытайтесь подключиться к серверу снова.')
            return (None,None)
        if show:
            print (data)
        (Gx,Gy)=tuple(map(int,re.search(r'(?<=G=\()\d+,\d+(?=\))',data).group(0).split(',')))
        p=int(re.search(r'(?<=E\(GF\()\d+(?=\)\))',data).group(0))
        group_order=int(re.search(r'(?<=order )\d+',data).group(0))
        axb=re.search(r'(?<=x\*\*3)([+-]\d+\*x)?([+-]\d+)?',data).group(0)
        ax=re.search(r'([+-]\d+\*x)?',axb)
        
        a=0 if ax==None else int(ax.group(0)[:-2])
        b=re.search(r'([+-]\d+)?$',axb)
        b=0 if b==None else int(b.group(0))
        (Qx,Qy)=tuple(map(int,re.search(r'(?<=Q=\()\d+,\d+(?=\))',data).group(0).split(',')))
        return (Gx,Gy,p,a,b,group_order,Qx,Qy)
    
    def checkSolution(self,k, show=True):
        self.s.sendall((str(k)+'\n').encode())
        data=self.recv_until(b'\n')
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Ошибка при декодировании Юникода. Попытайтесь подключиться к серверу снова.')
            return None
        if show:
            print (data)
        if data.find('flag')!=-1:
            return True
        else:
            data=self.recv_until(b'>')
            try:
                data=data.decode()
            except UnicodeDecodeError:
                print ('Ошибка при декодировании Юникода. Попытайтесь подключиться к серверу снова.')
                return None
            if show:
                print (data)
            return False
    def __del__(self):
        self.s.close()

vs=VulnServerClient()
(Gx,Gy,p,a,b,group_order,Qx,Qy)=vs.getChallenge()


Welcome to Elliptic Curve Subgroup Attack task
I will be using a generator G=(21326301780941263400978836898271986632548723234686720826883715272654258633762,33857809542043362783197813048043633104360814787025730443948009077920614105715) on E(GF(96071468671902143188546416236930573280881678549886695248698673777654734225923)) for y**2=x**3+2*x+9 which creates a group of order 21532617711820733
Q=kG
Q=(628967128461021131781469251586707446909833670052841805168064949059963263507,55276444660866149828911645283665301969409218974930995351013324734874947857734) on E(GF(96071468671902143188546416236930573280881678549886695248698673777654734225923)) for y**2=x**3+2*x+9
Find k and send it to me:
>


In [12]:
curve=EllipticCurve(p,a,b)
G=Point(curve,Gx,Gy)
Q=Point(curve,Qx,Qy)
#Можно складывать точки:
print (G+G)
#Можно умножать их на скаляр
print (G*group_order)

(8664881640451828217537330008063938984181319142460005167298024531664635858823,61795671233769785282270838840509334018357346659208038794225677950134535276639) на E(GF(96071468671902143188546416236930573280881678549886695248698673777654734225923)) для y**2=x**3+2*x+9
Точка на Бесконечности на E(GF(96071468671902143188546416236930573280881678549886695248698673777654734225923)) для y**2=x**3+2*x+9


In [13]:
vs.checkSolution(1)

Wrong k. Try again.

>


False

Factorization

In [14]:
def factorize(n):
    factors = {}
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors[d] = factors.get(d, 0) + 1
            n //= d
        d += 1
    if n > 1:
        factors[n] = 1
    return factors

BSGS

In [15]:
import math

def baby_step_giant_step(G, Y, order):
    m = int(math.isqrt(order)) + 1
    
    table = {}
    current = Point(G.curve, 0, 1, True)
    for j in range(m):
        key = (current.x, current.y, current.is_identity())
        if key not in table:
            table[key] = j
        current = current + G
        
    neg_m_G = G * (-m)
    current_y = Y
    
    for i in range(m):
        key = (current_y.x, current_y.y, current_y.is_identity())
        if key in table:
            j = table[key]
            return (i * m + j) % order
        current_y = current_y + neg_m_G
        
    raise ValueError("BSGS failed: no solution found")

Prime power

In [21]:
def solve_prime_power(G_sub, Q_sub, q, e):
    k_sub = 0
    gamma = G_sub * (q ** (e - 1))
    q_pow = 1
    
    for l in range(e):
        R = Q_sub + (-(G_sub * k_sub))
        
        exponent = q ** (e - 1 - l)
        target = R * exponent
        
        if q < 5000:
            print("Brute-force if...")
            cur = Point(G_sub.curve, 0, 1, True)
            k_l = None
            for val in range(q):
                if cur == target:
                    k_l = val
                    break
                cur = cur + gamma
            if k_l is None:
                raise ValueError("Brute-force failed")
        else:
            print("BSGS if...")
            k_l = baby_step_giant_step(gamma, target, q)
            
        k_sub += k_l * q_pow
        q_pow *= q
        
    return k_sub

CRT

In [22]:
def crt(remainders, moduli):
    M = 1
    for m in moduli:
        M *= m
        
    result = 0
    for r, m in zip(remainders, moduli):
        Mi = M // m
        inv = pow(Mi, -1, m)
        result = (result + r * Mi * inv) % M
    return result

Attack

In [23]:
def run_pohlig_hellman_ec(G, Q, order, factors):
    remainders = []
    moduli = []
    
    for q, e in factors.items():
        qe = q ** e
        ni = order // qe
        
        G_sub = G * ni
        Q_sub = Q * ni
        
        k_sub = solve_prime_power(G_sub, Q_sub, q, e)
        remainders.append(k_sub)
        moduli.append(qe)
        
    return crt(remainders, moduli)

Run

In [24]:
order = group_order
factors = factorize(order)
print(factors)

{7: 1, 211: 1, 91121: 1, 159991849: 1}


In [25]:
result = run_pohlig_hellman_ec(G=G, Q=Q, order=order, factors=factors)

Brute-force if...
Brute-force if...
BSGS if...
BSGS if...


In [26]:
print(result.bit_length())

55


In [27]:
vs.checkSolution(result)

Congratulations, your flag is: CRYPTOTRAINING{3ll1p71c_curv35_4r3_ju57_4n07h3r_7yp3_0f_gr0up}.



True